In [1]:
# 1. Configuración de la Variedad y Coordenadas
M = Manifold(4, 'M', latex_name=r'\mathcal{M}')
X.<t, r, th, ph> = M.chart(r't r th:\theta ph:\phi')

# 2. Parámetros del Fondo
var('M_m Q l', domain='real') # Usamos M_m para la masa y evitar conflicto con la variedad M
f = 1 - 2*M_m/r + Q^2/r^2

# 3. Métrica de Fondo (Reissner-Nordström)
g_bg = M.metric('g_bg')
g_bg[0,0] = -f
g_bg[1,1] = 1/f
g_bg[2,2] = r^2
g_bg[3,3] = r^2 * sin(th)^2

# 4. Potencial Electromagnético de Fondo
A_bg = M.one_form('A_bg')
A_bg[0] = -Q/r

# 5. Definición de las Funciones de Amplitud (dependen de t y r)
H0 = function('H_0')(t, r)
H1 = function('H_1')(t, r)
H2 = function('H_2')(t, r)
K = function('K')(t, r)
u_t = function('u_t')(t, r)
u_r = function('u_r')(t, r)

# Armónico Esférico Escalar (depende de theta)
Y = function('Y')(th)

# 6. Construcción del Tensor de Perturbación Métrica (h_mu_nu)
h = M.tensor_field(0, 2, 'h', sym=(0,1))
h[0,0] = -f * H0 * Y
h[0,1] = H1 * Y
h[1,1] = (1/f) * H2 * Y
h[2,2] = r^2 * K * Y
h[3,3] = r^2 * sin(th)^2 * K * Y

# 7. Construcción de la Perturbación del Cuadripotencial (a_mu)
a_pert = M.one_form('a_pert')
a_pert[0] = u_t * Y
a_pert[1] = u_r * Y

print("Configuración completada. Tensores h_mu_nu y a_mu construidos en el gauge de Regge-Wheeler y v=0.")

Configuración completada. Tensores h_mu_nu y a_mu construidos en el gauge de Regge-Wheeler y v=0.


In [2]:
# 1. Definición de las nuevas amplitudes radiales para paridad impar
h0 = function('h_0')(t, r)
h1 = function('h_1')(t, r)
a0 = function('a_0')(t, r)

# 2. Función angular para paridad impar: W(theta) = sin(theta) * d(P_l)/d(theta)
W = function('W')(th)

# 3. Construcción del Tensor de Perturbación Métrica AXIAL (h_ax)
h_ax = M.tensor_field(0, 2, 'h_ax', sym=(0,1))
# Solo sobreviven las componentes cruzadas con phi (índice 3)
h_ax[0, 3] = h0 * W  # Componente t-phi
h_ax[1, 3] = h1 * W  # Componente r-phi

# 4. Construcción del Cuadripotencial AXIAL (a_ax)
a_ax = M.one_form('a_ax')
# Solo sobrevive la componente acimutal
a_ax[3] = a0 * W

print("Ansatz Axial configurado: h_ax y a_ax construidos con 3 grados de libertad (h0, h1, a0).")

Ansatz Axial configurado: h_ax y a_ax construidos con 3 grados de libertad (h0, h1, a0).


In [3]:
# 1. Calculamos el tensor de Faraday perturbado (f = da)
# En SageManifolds, la derivada exterior se calcula con exterior_derivative()
f_pert = a_ax.exterior_derivative()
f_pert.set_name('f_pert', latex_name=r'f_{\mu\nu}')

print("Componentes no nulas del tensor de Faraday perturbado f_mu_nu:")
f_pert.display()

# 2. Tensor de Faraday del Fondo (F = dA_bg)
F_bg = A_bg.exterior_derivative()

# 3. Subimos índices para preparar las ecuaciones de campo
# Necesitamos f^mu_nu = g^{mu alpha} g^{nu beta} f_{alpha beta}
# Usamos la métrica de fondo g_bg para subir los índices de la perturbación (regla de primer orden)
g_inv = g_bg.inverse()
f_pert_up = f_pert.up(g_bg)

print("\nTensor Faraday perturbado contravariante f^{mu nu} calculado.")

Componentes no nulas del tensor de Faraday perturbado f_mu_nu:

Tensor Faraday perturbado contravariante f^{mu nu} calculado.


In [14]:
f_pert_up.display()

-W(th)*d(a_0)/dt/((1.0*Q^2 - 2.0*M_m*r + 1.0*r^2)*sin(th)^2) ∂/∂t⊗∂/∂ph + (1.0*Q^2 - 2.0*M_m*r + 1.0*r^2)*W(th)*d(a_0)/dr/(r^4*sin(th)^2) ∂/∂r⊗∂/∂ph + a_0(t, r)*d(W)/dth/(r^4*sin(th)^2) ∂/∂th⊗∂/∂ph + W(th)*d(a_0)/dt/((1.0*Q^2 - 2.0*M_m*r + 1.0*r^2)*sin(th)^2) ∂/∂ph⊗∂/∂t - (1.0*Q^2 - 2.0*M_m*r + 1.0*r^2)*W(th)*d(a_0)/dr/(r^4*sin(th)^2) ∂/∂ph⊗∂/∂r - a_0(t, r)*d(W)/dth/(r^4*sin(th)^2) ∂/∂ph⊗∂/∂th

In [6]:
# 1. Definimos la conexión de fondo (derivada covariante del fondo)
nabla = g_bg.connection()

# 2. Calculamos las derivadas covariantes del tensor métrico perturbado h_ax
nabla_h = nabla(h_ax)

# 3. Construimos delta Gamma (Símbolos de Christoffel linealizados)
# Fórmula: delta_Gamma^a_{bc} = 1/2 * g^{ad} ( nabla_b h_{dc} + nabla_c h_{bd} - nabla_d h_{bc} )
delta_Gamma = M.tensor_field(1, 2, 'delta_Gamma', sym=(1,2))
for a in M.irange():
    for b in M.irange():
        for c in range(b, 4): # Aprovechamos la simetría en los índices inferiores
            suma = 0
            for d in M.irange():
                suma += 0.5 * g_inv[a,d] * (nabla_h[b,d,c] + nabla_h[c,b,d] - nabla_h[d,b,c])
            delta_Gamma[a,b,c] = suma
            if b != c:
                delta_Gamma[a,c,b] = suma

print("Símbolos de Christoffel perturbados (delta_Gamma) calculados con éxito.")

# 4. Calculamos la derivada covariante de delta_Gamma
nabla_delta_Gamma = nabla(delta_Gamma)

# 5. Construimos el tensor de Ricci linealizado delta_R_{mu nu}
# Fórmula: delta_R_{ab} = nabla_c delta_Gamma^c_{ab} - nabla_b delta_Gamma^c_{ac}
delta_R = M.tensor_field(0, 2, 'delta_R', sym=(0,1))
for a in M.irange():
    for b in range(a, 4):
        suma1 = sum(nabla_delta_Gamma[c,c,a,b] for c in M.irange())
        suma2 = sum(nabla_delta_Gamma[b,c,a,c] for c in M.irange())
        delta_R[a,b] = suma1 - suma2
        if a != b:
            delta_R[b,a] = delta_R[a,b]

print("Tensor de Ricci linealizado (delta_R) calculado con éxito.")

# Mostrar qué componentes sobrevivieron (deberían ser solo cruzadas con phi)
print("\nComponentes no nulas de delta_R:")
for a in M.irange():
    for b in range(a, 4):
        if delta_R[a,b] != 0:
            print(f"delta_R[{a},{b}] es NO NULA.")

Símbolos de Christoffel perturbados (delta_Gamma) calculados con éxito.
Tensor de Ricci linealizado (delta_R) calculado con éxito.

Componentes no nulas de delta_R:
delta_R[0,3] es NO NULA.
delta_R[1,3] es NO NULA.
delta_R[2,3] es NO NULA.


In [9]:
# 1. Extraemos la componente t-phi (0,3) de delta_R
eq_03 = delta_R[0,3]

# 2. Simplificamos la expresión para ver su estructura radial y temporal
# Sage mostrará las derivadas de h_0, h_1 respecto a t y r.
print("Ecuación delta_R[0,3] (componente t-phi):")
eq_03.display()

Ecuación delta_R[0,3] (componente t-phi):


(t, r, th, ph) ↦ 1/2*(((Q^4*r^4 - 5*M_m*Q^2*r^5 - 3*M_m*r^7 + (6*M_m^2 + Q^2)*r^6)*h_0(t, r) - (Q^4*r^5 - 4*M_m*Q^2*r^6 - 4*M_m*r^8 + r^9 + 2*(2*M_m^2 + Q^2)*r^7)*d(h_0)/dr)*W(th)*sin(th)^4 - ((10*M_m*r^7 - 4*r^8 + (-4.0*M_m^2 - 5.0*Q^2 + 2.0)*r^6 - 2.0*Q^6 + 10.0*M_m*Q^4*r + (4*M_m*Q^2 - 2.0*M_m)*r^5 - (Q^4 + 8.0*M_m^2)*r^4 + (8.0*M_m^3 + 12.0*M_m*Q^2)*r^3 + (-16.0*M_m^2*Q^2 - 4.0*Q^4)*r^2)*h_0(t, r) + (-1.0*Q^2*r^5 + 2.0*M_m*r^6 - 1.0*r^7)*d(h_0)/dr)*W(th)*sin(th)^2 - 2*(Q^2*r^6 - 2*M_m*r^7 + r^8)*W(th)*h_0(t, r) - ((Q^2*r^6 - 2*M_m*r^7 + r^8)*cos(th)*h_0(t, r)*sin(th)^3 - (Q^2*r^6 - 2*M_m*r^7 + r^8)*cos(th)*h_0(t, r)*sin(th))*d(W)/dth)/((Q^2*r^8 - 2*M_m*r^9 + r^10)*sin(th)^4)

In [10]:
# 3. Extraemos la componente r-phi (1,3)
eq_13 = delta_R[1,3]
print("\nEcuación delta_R[1,3] (componente r-phi):")
eq_13.display()


Ecuación delta_R[1,3] (componente r-phi):


(t, r, th, ph) ↦ -1/2*((-1.0*r^9*d(h_0)/dt - (6*Q^4*r^4 - 21.0*M_m*Q^2*r^5 - 11.0*M_m*r^7 + 1.0*r^8 - (-18.0*M_m^2 - 7.0*Q^2)*r^6)*h_1(t, r) + (2.0*Q^4*r^5 - 8.0*M_m*Q^2*r^6 - 8.0*M_m*r^8 + 2.0*r^9 + (8.0*M_m^2 + 4.0*Q^2)*r^7)*d(h_1)/dr)*W(th)*sin(th)^4 + ((14*M_m*r^7 - 5*r^8 + (-8.0*M_m^2 - 7.0*Q^2 + 4.0)*r^6 - 2.0*Q^6 + 10.0*M_m*Q^4*r + (8*M_m*Q^2 - 10.0*M_m)*r^5 - (2*Q^4 + 8.0*M_m^2 - 6.0*Q^2)*r^4 + (8.0*M_m^3 + 12.0*M_m*Q^2)*r^3 + (-16.0*M_m^2*Q^2 - 4.0*Q^4)*r^2)*h_1(t, r) + (-1.0*Q^2*r^5 + 2.0*M_m*r^6 - 1.0*r^7)*d(h_0)/dt + (-2.0*Q^2*r^5 + 4.0*M_m*r^6 - 2.0*r^7)*d(h_1)/dr)*W(th)*sin(th)^2 + 2*(Q^2*r^6 - 2*M_m*r^7 + r^8)*W(th)*h_1(t, r) + ((Q^2*r^6 - 2*M_m*r^7 + r^8)*cos(th)*h_1(t, r)*sin(th)^3 - (Q^2*r^6 - 2*M_m*r^7 + r^8)*cos(th)*h_1(t, r)*sin(th))*d(W)/dth)/((Q^2*r^8 - 2*M_m*r^9 + r^10)*sin(th)^4)

In [12]:
# 1. Parámetros y tensores auxiliares
pi = var('pi')
F_bg_up = F_bg.up(g_bg)
f_pert_up = f_pert.up(g_bg)
h_ax_up = h_ax.up(g_bg)

# 2. Contracción escalar F_{ab} F^{ab} del fondo
F_sq_bg = sum(F_bg[a,b]*F_bg_up[a,b] for a in M.irange() for b in M.irange())

# 3. Construcción analítica de delta_T_{mu nu}
delta_T = M.tensor_field(0, 2, 'delta_T', sym=(0,1))

for mu in M.irange():
    for nu in range(mu, 4):
        # Termino 1: F_{mu alpha} * f_pert_{nu}^{alpha}
        term1 = sum(F_bg[mu, alpha] * sum(g_bg.inverse()[alpha, beta]*f_pert[nu, beta] for beta in M.irange()) for alpha in M.irange())
        # Termino 2: f_pert_{mu alpha} * F_{nu}^{alpha}
        term2 = sum(f_pert[mu, alpha] * sum(g_bg.inverse()[alpha, beta]*F_bg[nu, beta] for beta in M.irange()) for alpha in M.irange())
        # Termino 3: - F_{mu alpha} F_{nu beta} h^{alpha beta}
        term3 = - sum(F_bg[mu, alpha] * sum(F_bg[nu, beta] * h_ax_up[alpha, beta] for beta in M.irange()) for alpha in M.irange())
        # Termino 4: - 1/4 * h_{mu nu} * (F_bg^2)
        term4 = - 0.25 * h_ax[mu, nu] * F_sq_bg
        
        # Suma total (dividida por 4*pi)
        suma_total = (term1 + term2 + term3 + term4) / (4*pi)
        
        delta_T[mu, nu] = suma_total
        if mu != nu:
            delta_T[nu, mu] = suma_total

print("Tensor de Energía-Momento perturbado (delta_T) calculado.")
print("\nComponentes no nulas de delta_T:")
for mu in M.irange():
    for nu in range(mu, 4):
        if delta_T[mu,nu] != 0:
            print(f"delta_T[{mu},{nu}]:")
            #delta_T[mu,nu].display()
            print(delta_T[mu,nu].expr())
            print("---")

Tensor de Energía-Momento perturbado (delta_T) calculado.

Componentes no nulas de delta_T:
delta_T[0,3]:
1/4*(0.5*Q^2*h_0(t, r) + (Q^3 - 2*M_m*Q*r + Q*r^2)*diff(a_0(t, r), r))*W(th)/(pi*r^4)
---
delta_T[1,3]:
1/4*(Q*r^4*diff(a_0(t, r), t) + (0.5*Q^4 - 1.0*M_m*Q^2*r + 0.5*Q^2*r^2)*h_1(t, r))*W(th)/((Q^2*r^4 - 2*M_m*r^5 + r^6)*pi)
---


In [13]:
# === CÁLCULO DE LA ECUACIÓN DE MAXWELL PERTURBADA ===

# 1. Calculamos la derivada covariante del tensor de Faraday perturbado contravariante
# nabla_f_up tendrá índices [mu, nu, rho] que representa \nabla_\rho f^{\mu\nu}
nabla_f_up = nabla(f_pert_up)

# 2. Construimos el vector delta_M^mu
delta_M = M.vector_field('delta_M')

for mu in M.irange():
    # Termino 1: \nabla_\nu f^{\mu\nu} (Contraemos el índice 'nu' y la derivada 'nu')
    term1 = sum(nabla_f_up[mu, nu, nu] for nu in M.irange())
    
    # Termino 2: \delta\Gamma^\mu_{\nu\lambda} \mathring{F}^{\lambda\nu}
    # Ojo a la antisimetría del Faraday de fondo F_bg_up
    term2 = sum(delta_Gamma[mu, nu, lam] * F_bg_up[lam, nu] for nu in M.irange() for lam in M.irange())
    
    delta_M[mu] = term1 + term2

print("Ecuación de Maxwell linealizada (delta_M) calculada con éxito.")

# 3. Extraemos y mostramos la componente phi (índice 3)
eq_maxwell_phi = delta_M[3]

# Usamos .expr() para forzar a Sage a imprimir el monstruo simbólico 
# por si el .display() se queda en blanco
print("\nEcuación delta_M[3] (componente phi):")
print(eq_maxwell_phi.expr())

Ecuación de Maxwell linealizada (delta_M) calculada con éxito.

Ecuación delta_M[3] (componente phi):
-((-1.0*Q^6*r + 6.0*M_m*Q^4*r^2 + 6.0*M_m*r^6 - 1.0*r^7 + (-12.0*M_m^2 - 3.0*Q^2)*r^5 + (8.0*M_m^3 + 12.0*M_m*Q^2)*r^4 + (-12.0*M_m^2*Q^2 - 3.0*Q^4)*r^3)*a_0(t, r)*cos(th)*diff(W(th), th) + (1.0*Q^6*r - 6.0*M_m*Q^4*r^2 - 6.0*M_m*r^6 + 1.0*r^7 + (12.0*M_m^2 + 3.0*Q^2)*r^5 + (-8.0*M_m^3 - 12.0*M_m*Q^2)*r^4 + (12.0*M_m^2*Q^2 + 3.0*Q^4)*r^3)*a_0(t, r)*sin(th)*diff(W(th), th, th) - ((Q^4*r^5 - 4*M_m*Q^2*r^6 - 4*M_m*r^8 + r^9 + 2*(2*M_m^2 + Q^2)*r^7)*diff(a_0(t, r), t, t) - (-2.0*Q^8 + 14.0*M_m*Q^6*r + 2.0*M_m*r^7 + (-12.0*M_m^2 - 2.0*Q^2)*r^6 + (24.0*M_m^3 + 18.0*M_m*Q^2)*r^5 + (-16.0*M_m^4 - 48.0*M_m^2*Q^2 - 6.0*Q^4)*r^4 + (40.0*M_m^3*Q^2 + 30.0*M_m*Q^4)*r^3 + (-36.0*M_m^2*Q^4 - 6.0*Q^6)*r^2)*diff(a_0(t, r), r) - (1.0*Q^8*r - 8.0*M_m*Q^6*r^2 - 8.0*M_m*r^8 + 1.0*r^9 + (24.0*M_m^2 + 4.0*Q^2)*r^7 + (-32.0*M_m^3 - 24.0*M_m*Q^2)*r^6 + (16.0*M_m^4 + 48.0*M_m^2*Q^2 + 6.0*Q^4)*r^5 + (-32.0*M_m^3*Q